In [1]:
# import libraries and define functions
import numpy as np
import pandas as pd

# extract of landmarks from excel file
def transform_point_entry_to_array_list(df,grid,index_name):
    landmarks = df[df['grid']==grid][index_name].tolist()
    landmarks = [np.fromstring(pts.strip('[]\n'),sep=' ') for pts in landmarks]
    return landmarks
# extract matrix from excel file
def transform_matrix_entry_to_matrix(df,grid,column):
    matrix = df[df['grid']==grid][column].tolist()[0]
    matrix = np.fromstring(matrix.replace('[','').replace(']',''),sep=' ').reshape(3,3)
    return np.hstack([matrix[:2,:2].reshape(-1),matrix[:2,2]])
# extract matrix from excel file
# calculate the covariance ('error') of the matrix parameter
def calculate_parameter_covariance(pts_src,residuals):
    pts_src = np.asarray(pts_src)
    residuals = np.asarray(residuals)
    n = pts_src.shape[0]
    J = np.zeros((2*n, 6))
    for i, (xs, ys) in enumerate(pts_src):
        J[2*i]   = [xs, ys, 0, 0, 1, 0]
        J[2*i+1] = [0, 0, xs, ys, 0, 1]
    dof = max(1, 2*n - 6)  # degrees of freedom
    sigma2_hat = (residuals @ residuals) / dof
    # Covariance of parameters: sigma2 * (J^T J)^{-1}
    # Use pseudo-inverse in case J^T J near-singular
    JTJ = J.T.dot(J)
    inv_JTJ = np.linalg.pinv(JTJ)
    Sigma_p = sigma2_hat * inv_JTJ
    return Sigma_p
# transform a point into specific format for calculation
def Jp_affine_at_point(x):
    x1, x2 = x
    Jp = np.array([[x1, x2, 0, 0, 1, 0],
                   [0, 0, x1, x2, 0, 1]])
    return Jp
# Propagate a point trough all transforms and also propagate the error along with it
def propagate_point_A_to_D(xA, affine1, Sigma_p1, translation, affine2, Sigma_p2):
    ## define matrices
    # first affine transformation
    A1 = affine1[:4].reshape(2,2)
    t1 = affine1[4:]
    # translation
    Ts = translation[:4].reshape(2,2)
    ts = translation[4:]
    # second affine transformation
    A2 = affine2[:4].reshape(2,2)
    t2 = affine2[4:]

    ## propagate the point
    # first transform
    xB = A1.dot(xA) + t1
    # second transform
    xC = Ts.dot(xB) + ts
    # third transform
    xD = A2.dot(xC) + t2

    ## propagate the error
    # first transform
    Jp1 = Jp_affine_at_point(xA)
    Sigma_B = Jp1.dot(Sigma_p1).dot(Jp1.T)
    # second transform (no propagation only scaling)
    Sigma_C = Ts.dot(Sigma_B).dot(Ts.T)
    # third transform
    # uncertainty just for last last transform
    Jp2 = Jp_affine_at_point(xC)
    Sigma_from_CD_params = Jp2.dot(Sigma_p2).dot(Jp2.T)
    # actual propagation
    Sigma_from_prev = A2.dot(Sigma_C).dot(A2.T)
    # combined error
    Sigma_D = Sigma_from_CD_params + Sigma_from_prev
    return xD, Sigma_D

In [ ]:
# import excel files with landmarks and transformation matrices and points to transform
from pathlib import Path
path_server = Path('path/to/server')
file_landmarks = path_server/'sub/path/to/landmarks.xlsx'
data_landmarks = pd.read_excel(file_landmarks)
data_landmarks = data_landmarks.iloc[:-1]
file_afm_points = path_server/'sub/path/to/AFMCoords_df.xlsx'
data_afm_points = pd.read_excel(file_afm_points)

In [3]:
# get the different sections
sections = data_landmarks['section'].unique()

In [4]:
# iterate over each section and grid to get matrix parameter covariance
cov_p = dict()
matrices = dict()
tm_matrices = dict()
for section in sections:
    sub_cov_p = dict()
    sub_matrices = dict()
    sub_translation = dict()
    df = data_landmarks[data_landmarks['section']==section]
    grids = df['grid'].unique()
    for grid in grids:
        matrix = transform_matrix_entry_to_matrix(df,grid,'Matrix')
        pts_src = transform_point_entry_to_array_list(df,grid,'to_be_transformed')
        residuals = transform_point_entry_to_array_list(df,grid,'difference_single')
        residuals = np.hstack(residuals)
        sub_cov_p[grid] = calculate_parameter_covariance(pts_src,residuals)
        sub_matrices[grid] = matrix
        if grid != 'vis':
            matrix = transform_matrix_entry_to_matrix(df,grid,'tm_matrix')
            sub_translation[grid] = matrix
    cov_p[section] = sub_cov_p
    matrices[section] = sub_matrices
    tm_matrices[section] = sub_translation

In [5]:
df = []
for section in sections:
    afm_df = data_afm_points[data_afm_points['section']==section]
    grids = afm_df['grid'].unique().tolist()
    for grid in grids:
        # propably for your data
        coords = transform_point_entry_to_array_list(afm_df,grid,'AFM_grid_coords')
        for xA in coords:
            affine1 = matrices[section][grid]
            Sigma_p1 = cov_p[section][grid]
            translation = tm_matrices[section][grid]
            affine2 = matrices[section]['vis']
            Sigma_p2 = cov_p[section]['vis']
            xt,sigma_D = propagate_point_A_to_D(xA, affine1, Sigma_p1, translation, affine2, Sigma_p2)
            std_x, std_y = np.sqrt(np.diag(sigma_D))
            df.append(pd.DataFrame({'section':[section],'grid':[grid],
                                    'x_start':xA[0],'y_start':xA[1],
                                    'x_final':[xt[0]],'y_final':[xt[1]],'std_x':[std_x],'std_y':[std_y]}))
df = pd.concat(df)
df

,section,grid,x_start,y_start,x_final,y_final,std_x,std_y
0,LM4427,grid01,1293.703326,1293.703326,6747.351208,8404.381753,12.413459,12.402521
0,LM4427,grid01,1293.703326,1233.769254,6767.650420,8419.770994,12.409286,12.400154
0,LM4427,grid01,1293.703326,1173.835181,6787.949631,8435.160234,12.409481,12.401936
0,LM4427,grid01,1293.703326,1113.901109,6808.248843,8450.549475,12.414042,12.407865
0,LM4427,grid01,1293.703326,1053.967036,6828.548055,8465.938716,12.422965,12.417934
...,...,...,...,...,...,...,...,...
0,ACH74LMA,grid03,754.296674,994.032964,8510.889510,3096.416432,496.467887,496.465418
0,ACH74LMA,grid03,754.296674,1053.967036,8536.158819,3094.625986,498.408581,498.406137
0,ACH74LMA,grid03,754.296674,1113.901109,8561.428128,3092.835539,500.349462,500.347026
0,ACH74LMA,grid03,754.296674,1233.769254,8611.966746,3089.254647,504.231774,504.229312
